# The hybrid pipeline

**Module 2 -- Lesson 10**

Over the last several lessons, you built four parsing tools: regex templates, a finetuned NER model, an LLM prompt, and a zero-shot extractor (GLiNER2). Each has strengths on different kinds of input.

This notebook combines three of them into a pipeline. GLiNER2 was useful for exploration, but the finetuned NER model gives more control -- it's trained on your labels and runs consistently on your data.

1. **Templates** -- identify which fields exist and where the header boundaries are
2. **NER** -- run on every template-matched header to identify individual people and email addresses within the raw field strings
3. **LLM** -- for the emails templates couldn't handle (from notebook 2.6), where entities are already split into arrays
4. **CSV output** -- node and relationship files ready for cleaning and import

## Setup

In [16]:
%pip install spacy python-dateutil -q

Note: you may need to restart the kernel to use updated packages.


In [17]:
import json
import csv
import re
from pathlib import Path
from collections import Counter

import spacy
from dateutil import parser as dateparser
from helpers.enron_templates import strip_enron_boilerplate, extract_enron_headers

TXT_DIR = Path("data/extracted_text")
txt_files = sorted(TXT_DIR.glob("*.txt"))
EMAIL_RE = re.compile(r'[\w.+\-]+@[\w.\-]+\.\w+')

nlp_ner = spacy.load("data/ner_models/ner_blank/model-best")

print(f"{len(txt_files)} text files")
print(f"NER labels: {nlp_ner.get_pipe('ner').labels}")

4911 text files
NER labels: ('CC_RECIPIENT', 'CC_RECIPIENT_EMAIL', 'DATE', 'RECIPIENT', 'RECIPIENT_EMAIL', 'SENDER', 'SENDER_EMAIL', 'SUBJECT')


## 1. Template pass

Run every email through the template parser. Templates identify which fields exist and where the header ends — but the field values are raw strings that may contain multiple people, email addresses, and noise.

In [18]:
template_results = {}
template_failures = []

for txt_path in txt_files:
    doc_id = txt_path.stem
    text = txt_path.read_text(encoding="utf-8")
    result = extract_enron_headers(text)

    if result:
        template_results[doc_id] = result
    else:
        template_failures.append(doc_id)

print(f"Template matched: {len(template_results):,}")
print(f"Template failed:  {len(template_failures)}")

Template matched: 4,841
Template failed:  70


## 2. NER pass

The NER model runs on the header text of every template-matched email. It identifies individual entity spans — each person's name, each email address, the date, and the subject — tagged with their role (SENDER, RECIPIENT, CC_RECIPIENT, etc.).

Entities come out in document order: a RECIPIENT name is immediately followed by its RECIPIENT_EMAIL. The record builder walks this sequence and pairs each name with its email, creating one entry per person rather than separate orphaned names and addresses.

In [19]:
ner_docs = {}  # doc_id -> spaCy Doc object (preserves entity order)

for doc_id, template in template_results.items():
    text = (TXT_DIR / f"{doc_id}.txt").read_text(encoding="utf-8")
    stripped = strip_enron_boilerplate(text)
    header = "\n".join(stripped.split("\n")[:20])
    ner_docs[doc_id] = nlp_ner(header)

print(f"NER processed: {len(ner_docs):,} emails")

# Show one example
sample_id = list(ner_docs.keys())[0]
print(f"\nExample: {sample_id}")
for ent in ner_docs[sample_id].ents:
    print(f"  {ent.label_:20s} -> {ent.text!r}")

NER processed: 4,841 emails

Example: E0000813E
  SENDER               -> 'Rob Bradley'
  SENDER_EMAIL         -> 'rob.bradley@enron.com'
  DATE                 -> 'Tue, 21 Nov 2000 08:34:00 -0800 (PST)'
  RECIPIENT            -> 'Kenneth Lay'
  RECIPIENT_EMAIL      -> 'kenneth.lay@enron.com'
  CC_RECIPIENT         -> 'Alhamd Alkhayat'
  RECIPIENT_EMAIL      -> 'alhamd.alkhayat@enron.com'
  SUBJECT              -> 'Remarks to EES Employees--December 1'


## 3. LLM results

The 70 template failures were sent to the LLM in notebook 2.6. The LLM already returns individuals in arrays (`RECIPIENT_NAMES: ["Alice", "Bob"]`), so no further splitting is needed.

In [20]:
llm_results = {}

with open("data/all_parsed_records.json") as f:
    all_parsed = json.load(f)

for record in all_parsed:
    if record.get("_template") == "llm":
        doc_id = record["doc_id"]
        llm_results[doc_id] = record

print(f"LLM results loaded: {len(llm_results)}")

LLM results loaded: 70


## 4. Build records

Assemble every email into a standardised record. For template-matched emails, the NER entities are the source of names and emails. For LLM emails, the parsed arrays are the source.

The key insight for NER records: entities come out in **document order**. When you read the header top to bottom, a RECIPIENT name is immediately followed by its RECIPIENT_EMAIL. The `pair_entities` function walks this sequence and pairs each name with the email that follows it. If a bare email appears without a preceding name (like `susan.j.mara@enron.com`), it becomes an email-only entry.

The template's `sent` field provides the raw date string (NER captures it too, but the template value is more reliable for date parsing).

In [21]:
def extract_domain(email):
    if email and '@' in email:
        return email.split('@')[-1].lower().strip('.')
    return None


def parse_date(date_raw):
    if not date_raw or not date_raw.strip():
        return None
    try:
        return dateparser.parse(date_raw, fuzzy=True).isoformat()
    except (ValueError, OverflowError):
        return None

In [22]:
def pair_entities(doc):
    """Walk NER entities in document order and pair names with their emails.
    
    Returns a dict with sender, recipients, cc, date, subject — each recipient
    entry has both name and email where available.
    """
    sender_name = None
    sender_email = None
    date = None
    subject = None
    recipients = []
    cc = []
    
    pending_name = None
    pending_type = None
    
    for ent in doc.ents:
        label = ent.label_
        text = ent.text
        
        if label == "SENDER":
            sender_name = text
        elif label == "SENDER_EMAIL":
            sender_email = text.lower()
        elif label == "DATE":
            date = text
        elif label == "SUBJECT":
            subject = text
        
        elif label == "RECIPIENT":
            if pending_name:
                target = cc if pending_type == "cc" else recipients
                target.append({"name": pending_name, "email": None, "type": pending_type})
            pending_name = text
            pending_type = "to"
        
        elif label == "RECIPIENT_EMAIL":
            if pending_name and pending_type:
                target = cc if pending_type == "cc" else recipients
                target.append({"name": pending_name, "email": text.lower(), "type": pending_type})
                pending_name = None
            else:
                recipients.append({"name": None, "email": text.lower(), "type": "to"})
        
        elif label == "CC_RECIPIENT":
            if pending_name:
                target = cc if pending_type == "cc" else recipients
                target.append({"name": pending_name, "email": None, "type": pending_type})
            pending_name = text
            pending_type = "cc"
        
        elif label == "CC_RECIPIENT_EMAIL":
            if pending_name and pending_type == "cc":
                cc.append({"name": pending_name, "email": text.lower(), "type": "cc"})
                pending_name = None
            else:
                cc.append({"name": None, "email": text.lower(), "type": "cc"})
    
    if pending_name:
        target = cc if pending_type == "cc" else recipients
        target.append({"name": pending_name, "email": None, "type": pending_type})
    
    return {
        "sender_name": sender_name,
        "sender_email": sender_email,
        "date": date,
        "subject": subject,
        "recipients": recipients + cc,
    }


def get_body(doc_id, template):
    """Extract body text from the stripped email using the template's body index."""
    body_idx = template.get("_body_start_idx")
    if body_idx is None:
        return ""
    text = (TXT_DIR / f"{doc_id}.txt").read_text(encoding="utf-8")
    stripped = strip_enron_boilerplate(text)
    lines = stripped.split("\n")
    if body_idx < len(lines):
        return "\n".join(lines[body_idx:]).strip()
    return ""


def build_record_ner(doc_id, template, doc):
    """Build a record from template fields + NER entity pairing."""
    paired = pair_entities(doc)
    sender_email = paired["sender_email"]
    
    return {
        "doc_id": doc_id,
        "sender_name": paired["sender_name"],
        "sender_email": sender_email,
        "sender_domain": extract_domain(sender_email),
        "recipients": paired["recipients"],
        "date_raw": template.get("sent", ""),
        "date_parsed": parse_date(template.get("sent", "")),
        "subject": paired["subject"] or template.get("subject"),
        "body": get_body(doc_id, template),
        "method": "template+ner",
    }

In [23]:
def build_record_llm(doc_id, llm):
    """Build a record from LLM results (arrays or comma-joined strings)."""
    recipients = []

    # to/cc may be arrays (from 2.6 merge) or strings (legacy format)
    to_values = llm.get("to", [])
    if isinstance(to_values, str):
        to_values = [n.strip() for n in to_values.split(",") if n.strip()]
    for name in to_values:
        email = EMAIL_RE.search(name)
        if email:
            recipients.append({"name": None, "email": email.group().lower(), "type": "to"})
        else:
            recipients.append({"name": name, "email": None, "type": "to"})

    cc_values = llm.get("cc", [])
    if isinstance(cc_values, str):
        cc_values = [n.strip() for n in cc_values.split(",") if n.strip()]
    for name in cc_values:
        email = EMAIL_RE.search(name)
        if email:
            recipients.append({"name": None, "email": email.group().lower(), "type": "cc"})
        else:
            recipients.append({"name": name, "email": None, "type": "cc"})

    sender_name = llm.get("from", "") or None
    sender_email = None
    if sender_name:
        m = EMAIL_RE.search(sender_name)
        if m:
            sender_email = m.group().lower()
            sender_name = EMAIL_RE.sub('', sender_name).strip() or None

    # Body: read from extracted text, skip boilerplate + header
    body = ""
    txt_path = TXT_DIR / f"{doc_id}.txt"
    if txt_path.exists():
        text = txt_path.read_text(encoding="utf-8")
        stripped = strip_enron_boilerplate(text)
        lines = stripped.split("\n")
        for i, line in enumerate(lines):
            if line.strip().lower().startswith("subject"):
                body_start = i + 2 if i + 1 < len(lines) else i + 1
                body = "\n".join(lines[body_start:]).strip()
                break

    return {
        "doc_id": doc_id,
        "sender_name": sender_name,
        "sender_email": sender_email,
        "sender_domain": extract_domain(sender_email),
        "recipients": recipients,
        "date_raw": llm.get("sent", ""),
        "date_parsed": parse_date(llm.get("sent", "")),
        "subject": llm.get("subject"),
        "body": body,
        "method": "llm",
    }

## 5. Run the pipeline

In [24]:
records = []

for doc_id, template in template_results.items():
    doc = ner_docs.get(doc_id)
    if doc:
        records.append(build_record_ner(doc_id, template, doc))

for doc_id, llm in llm_results.items():
    records.append(build_record_llm(doc_id, llm))

covered = {r["doc_id"] for r in records}
uncovered = [d for d in template_failures if d not in covered]
if uncovered:
    print(f"Warning: {len(uncovered)} emails have no parse result")

print(f"Total records: {len(records):,}")
print(f"  template+ner: {sum(1 for r in records if r['method'] == 'template+ner'):,}")
print(f"  llm:          {sum(1 for r in records if r['method'] == 'llm')}")

Total records: 4,911
  template+ner: 4,841
  llm:          70


## 6. Validate

In [25]:
print("Null rates:")
for field in ["sender_name", "sender_email", "sender_domain", "subject", "date_parsed"]:
    null_count = sum(1 for r in records if not r.get(field))
    print(f"  {field:15s} {null_count:,} ({null_count/len(records)*100:.1f}%)")

print()
print("Recipient counts:")
total_recip = sum(len(r["recipients"]) for r in records)
with_name = sum(1 for r in records for p in r["recipients"] if p["name"])
with_email = sum(1 for r in records for p in r["recipients"] if p["email"])
print(f"  Total recipient entries: {total_recip:,}")
print(f"  With name:  {with_name:,}")
print(f"  With email: {with_email:,}")

print()
domains = Counter(r["sender_domain"] for r in records if r["sender_domain"])
print("Top 10 sender domains:")
for domain, count in domains.most_common(10):
    print(f"  {domain:30s} {count:,}")

Null rates:
  sender_name     965 (19.6%)
  sender_email    979 (19.9%)
  sender_domain   987 (20.1%)
  subject         0 (0.0%)
  date_parsed     71 (1.4%)

Recipient counts:
  Total recipient entries: 15,614
  With name:  12,515
  With email: 14,111

Top 10 sender domains:
  enron.com                      3,235
  hotmail.com                    16
  aol.com                        15
  nymex.com                      14
  haas.berkeley.edu              12
  txu.com                        12
  columbiaenergygroup.com        10
  carrfut.com                    10
  yahoo.com                      8
  caiso.com                      7


## 7. Write to CSV

Each node type and relationship type gets its own file.

**Nodes:** `emails.csv`, `users.csv`, `mailboxes.csv`, `domains.csv`

**Relationships:** `SENT.csv`, `RECEIVED.csv`, `CC_ON.csv`, `USED.csv`, `HAS_MAILBOX.csv`

In [26]:
def sanitise(val):
    """Strip embedded quotes and carriage returns from raw values."""
    return val.replace('"', '').replace('\r', '').strip() if val else ''

out_dir = Path("data/csv")
out_dir.mkdir(parents=True, exist_ok=True)

users = set()
mailboxes = set()
domains = set()
used_rels = set()
has_mailbox_rels = set()
sent_rels = []
received_rels = []
cc_on_rels = []

for r in records:
    sn = sanitise(r.get("sender_name") or "")
    se = sanitise(r.get("sender_email") or "")
    sd = r.get("sender_domain")

    if sn: users.add(sn)
    if se: mailboxes.add(se)
    if sn and se: used_rels.add((sn, se))
    if sd and se:
        domains.add(sd)
        has_mailbox_rels.add((sd, se))
    sent_rels.append({"doc_id": r["doc_id"], "name": sn, "address": se})

    for recip in r["recipients"]:
        name = sanitise(recip.get("name") or "")
        email = sanitise(recip.get("email") or "")
        domain = extract_domain(email) if email else None
        if name: users.add(name)
        if email: mailboxes.add(email)
        if domain:
            domains.add(domain)
            has_mailbox_rels.add((domain, email))
        if name and email: used_rels.add((name, email))
        rel = {"doc_id": r["doc_id"], "name": name, "address": email}
        if recip["type"] == "to":
            received_rels.append(rel)
        elif recip["type"] == "cc":
            cc_on_rels.append(rel)

def write_nodes(filename, header, rows):
    with open(out_dir / filename, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        for row in rows:
            w.writerow(row)

with open(out_dir / "emails.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["doc_id", "subject", "date_raw", "date_parsed", "method", "body"])
    w.writeheader()
    for r in records:
        w.writerow({
            "doc_id": r["doc_id"],
            "subject": sanitise(r["subject"] or ""),
            "date_raw": r["date_raw"],
            "date_parsed": r["date_parsed"] or "",
            "method": r["method"],
            "body": sanitise(r.get("body", "")),
        })

write_nodes("users.csv", ["name"], [[n] for n in sorted(users)])
write_nodes("mailboxes.csv", ["address"], [[a] for a in sorted(mailboxes)])
write_nodes("domains.csv", ["name"], [[d] for d in sorted(domains)])

def write_rels(filename, rels):
    with open(out_dir / filename, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["doc_id", "name", "address"])
        w.writeheader()
        for rel in rels:
            w.writerow(rel)

write_rels("SENT.csv", sent_rels)
write_rels("RECEIVED.csv", received_rels)
write_rels("CC_ON.csv", cc_on_rels)
write_nodes("USED.csv", ["name", "address"], sorted(used_rels))
write_nodes("HAS_MAILBOX.csv", ["domain", "address"], sorted(has_mailbox_rels))

print("Nodes:")
print(f"  emails.csv:       {len(records):,}")
print(f"  users.csv:        {len(users):,}")
print(f"  mailboxes.csv:    {len(mailboxes):,}")
print(f"  domains.csv:      {len(domains):,}")
print()
print("Relationships:")
print(f"  SENT.csv:         {len(sent_rels):,}")
print(f"  RECEIVED.csv:     {len(received_rels):,}")
print(f"  CC_ON.csv:        {len(cc_on_rels):,}")
print(f"  USED.csv:         {len(used_rels):,}")
print(f"  HAS_MAILBOX.csv:  {len(has_mailbox_rels):,}")

Nodes:
  emails.csv:       4,911
  users.csv:        5,552
  mailboxes.csv:    5,654
  domains.csv:      1,098

Relationships:
  SENT.csv:         4,911
  RECEIVED.csv:     14,266
  CC_ON.csv:        1,348
  USED.csv:         5,043
  HAS_MAILBOX.csv:  5,588


Open the CSV files in `data/csv/` to inspect. Names and emails are separate entries from NER — no unsplit multi-person strings. The next notebook cleans and normalizes these values before import.

## Summary

- **Templates** identify field boundaries for 98.5% of emails
- **NER** identifies individual entities within those fields — each person's name and email as a separate span
- **LLM** handles the 70 emails with OCR-garbled headers, returning individuals in arrays
- Names and emails are separate entries — no regex splitting needed, the NER handles `Last, First` names and mixed bare-email/name lists
- **CSV output** provides a checkpoint before cleaning and import

**Next:** Cleaning and normalization, then Neo4j import.